In [1]:
import os
import random
import shutil


base_dir = r"C:\Users\prach\Desktop\minor"  
images_dir = os.path.join(base_dir, "images")
labels_dir = os.path.join(base_dir, "labels")

output_dir = base_dir  
split_ratios = {"train": 0.8, "valid": 0.1, "test": 0.1}
image_exts = [".jpg", ".jpeg", ".png"]  


image_files = [f for f in os.listdir(images_dir) if os.path.splitext(f)[1].lower() in image_exts]
random.shuffle(image_files)


total = len(image_files)
train_end = int(split_ratios["train"] * total)
valid_end = train_end + int(split_ratios["valid"] * total)

splits = {
    "train": image_files[:train_end],
    "valid": image_files[train_end:valid_end],
    "test": image_files[valid_end:]
}


for split, files in splits.items():
    img_out = os.path.join(output_dir, split, "images")
    lbl_out = os.path.join(output_dir, split, "labels")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    for img_file in files:
        shutil.copy2(os.path.join(images_dir, img_file), os.path.join(img_out, img_file))

        label_file = os.path.splitext(img_file)[0] + ".txt"
        label_src = os.path.join(labels_dir, label_file)
        label_dst = os.path.join(lbl_out, label_file)

        if os.path.exists(label_src):
            shutil.copy2(label_src, label_dst)
        else:
            print(f"Missing label for {img_file}")

print("Dataset split complete")


Dataset split complete


In [2]:
base_dir = r"C:\Users\prach\Desktop\minor"
splits = ["train", "valid", "test"]

for split in splits:
    img_dir = os.path.join(base_dir, split, "images")
    lbl_dir = os.path.join(base_dir, split, "labels")

    img_count = len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    lbl_count = len([f for f in os.listdir(lbl_dir) if f.lower().endswith('.txt')])

    print(f"{split.upper()}: {img_count} images, {lbl_count} labels")


TRAIN: 23840 images, 23840 labels
VALID: 2980 images, 2980 labels
TEST: 2980 images, 2980 labels


In [3]:
import os

labels_root = r"C:\Users\prach\Desktop\minor"
splits = ["train", "valid", "test"]
num_classes = 11

def is_valid_line(line):
    parts = line.strip().split()
    if len(parts) != 5:
        return False
    try:
        class_id = int(parts[0])
        coords = list(map(float, parts[1:]))
        if not (0 <= class_id < num_classes):
            return False
        if not all(0 <= val <= 1 for val in coords):
            return False
    except ValueError:
        return False
    return True

for split in splits:
    lbl_dir = os.path.join(labels_root, split, "labels")
    for file in os.listdir(lbl_dir):
        if file.endswith(".txt"):
            path = os.path.join(lbl_dir, file)
            with open(path, "r") as f:
                lines = f.readlines()
                for i, line in enumerate(lines):
                    if not is_valid_line(line):
                        print(f"Invalid format in {file} (line {i+1}): {line.strip()}")


In [4]:
pip install ultralytics



   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 6.5 MB/s  0:00:00
   ---------------------------------------- 0.0/241.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/241.3 MB 6.1 MB/s eta 0:00:40
   ---------------------------------------- 2.6/241.3 MB 6.3 MB/s eta 0:00:38
    --------------------------------------- 3.7/241.3 MB 6.1 MB/s eta 0:00:40
    --------------------------------------- 5.0/241.3 MB 6.3 MB/s eta 0:00:38
   - -------------------------------------- 6.6/241.3 MB 6.3 MB/s eta 0:00:38
   - -------------------------------------- 7.9/241.3 MB 6.4 MB/s eta 0:00:37
   - -------------------------------------- 9.2/241.3 MB 6.4 MB/s eta 0:00:37
   - -------------------------------------- 10.5/241.3 MB 6.4 MB/s eta 0:00:36
   - -------------------------------------- 11.8/241.3 MB 6.5 MB/s eta 0:00:36
   -- ------------------------------------- 13.4/241.3 MB 6.5 MB/s eta 0:00:36
   

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  

model.train(
    data=r"C:\Users\prach\Desktop\minor\data.yaml",
    epochs=50,
    imgsz=512,
    batch=16,
    project="minor_project",
    name="yolov8n_training"
)


Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\prach\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.203  Python-3.12.11 torch-2.8.0+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\prach\Desktop\minor\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, 

In [5]:
from collections import Counter
import os

base_dir = r"C:\Users\prach\Desktop\minor"
splits = ["train", "valid", "test"]
class_counts = Counter()

for split in splits:
    lbl_dir = os.path.join(base_dir, split, "labels")
    if not os.path.exists(lbl_dir):
        print(f"Label folder not found: {lbl_dir}")
        continue

    for file in os.listdir(lbl_dir):
        if file.endswith(".txt"):
            with open(os.path.join(lbl_dir, file), "r") as f:
                for line in f:
                    class_id = line.strip().split()[0]
                    class_counts[class_id] += 1

print("Class Distribution Across All Splits:")
for class_id, count in sorted(class_counts.items(), key=lambda x: int(x[0])):
    print(f"Class {class_id}: {count} instances")


Class Distribution Across All Splits:
Class 0: 3704 instances
Class 1: 127873 instances
Class 2: 21491 instances
Class 3: 5101 instances
Class 4: 10838 instances
Class 5: 614 instances
Class 6: 13673 instances
Class 7: 3482 instances
Class 8: 541 instances
Class 9: 28 instances
Class 10: 7194 instances


In [1]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  

model.train(
    data=r"C:\Users\prach\Desktop\minor\data.yaml",
    epochs=10,
    imgsz=512,
    batch=16,
    project="minor_project",
    name="yolov8n_training"
)

Ultralytics 8.3.203  Python-3.12.11 torch-2.8.0+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\prach\Desktop\minor\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_training, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, p

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000025D86A42F60>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,  

In [2]:
model.val()


Ultralytics 8.3.203  Python-3.12.11 torch-2.8.0+cpu CPU (AMD Ryzen 5 5600H with Radeon Graphics)
Model summary (fused): 72 layers, 3,007,793 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 35.44.8 MB/s, size: 39.7 KB)
val: Scanning C:\Users\prach\Desktop\minor\valid\labels.cache... 2980 images, 366 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2980/2980 1.5Mit/s 0.0s
val: C:\Users\prach\Desktop\minor\valid\images\1478897760163798179_jpg.rf.98623be50b02ff17d58f89fddf7a0c6c.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 187/187 0.8it/s 3:451.2ss
                   all       2980      19687      0.782      0.528      0.578      0.317
                 biker        257        413       0.68      0.542      0.595      0.314
                   car       2566      13037       0.82      0.784      0.835      0.539
            pedestrian        702       2129      0.645   

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000025D9FB82A80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,  

In [3]:
results = model.predict(source=r"C:\Users\prach\Desktop\minor\test\images", save=True)



WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

image 1/2980 C:\Users\prach\Desktop\minor\test\images\1478019953180167674_jpg.rf.8a816c9d7e9b423a63ed6ecd4a663e47.jpg: 512x512 2 cars, 1 truck, 237.1ms
image 2/2980 C:\Users\prach\Desktop\minor\test\images\1478019955185244088_jpg.rf.gJpj2eCO1Dd7Sic9WlhE.jpg: 512x512 3 cars, 1 pedestrian, 2 trucks, 74.7ms
image 3/2980 C:\Users\prach\Desktop\minor\test\images\1478019961182003465_jpg.rf.1YoPELIQZqEgJpEQITU3.jpg: 512x512 2 cars, 76.6ms
image 4/2980 C:\Users\p